In [1]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import wbgapi as wb


COUNTRIES_CSV_PATH = r"/workspace/qgi\qgi-4.0\data\countries.csv"

def read_parquet_as_df(parquet_path: str):
    return pq.read_table(parquet_path).to_pandas()

def preprocess_data(df):
    """
    Preprocesses data for each country by handling NaN values and organizing data for analysis.
    
    This function processes the input DataFrame to prepare data for each country:
    1. Filters the data for each country
    2. Sorts the data by year
    3. Handles NaN values by either skipping countries with more than one NaN or interpolating single NaN values
    4. Creates a list of dictionaries containing the country name, original data, and cleaned values
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    
    Returns:
    list: A list of dictionaries, where each dictionary contains:
        - "country": The country name
        - "data": The original country data sorted by year
        - "values": Cleaned numeric values with NaNs handled
    """
    countries = df["country"].unique()
    preprocessed_data = []

    for country in countries:
        country_data = df[df["country"] == country].sort_values("year")
        values = country_data["value"].values.copy()

        # Handle NaN values
        nan_indices = np.isnan(values)
        if np.sum(nan_indices) > 1:
            continue

        if np.sum(nan_indices) == 1:
            nan_pos = np.where(nan_indices)[0][0]
            if nan_pos == 0:
                values[nan_pos] = values[nan_pos + 1]
            elif nan_pos == len(values) - 1:
                values[nan_pos] = values[nan_pos - 1]
            else:
                values[nan_pos] = (values[nan_pos - 1] + values[nan_pos + 1]) / 2

        preprocessed_data.append({
            "country": country,
            "data": country_data,
            "values": values
        })

    return preprocessed_data

def calculate_total_comparisons(df, min_length=3, max_length=5):
    """
    Calculate the total number of segment comparisons for progress tracking.
    
    This function computes the total number of comparisons that would be made 
    during the pattern extraction process. It considers all possible segment 
    lengths within the specified range and all country pairs.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame containing indicator data for multiple countries
    min_length (int): Minimum segment length to consider
    max_length (int): Maximum segment length to consider
    
    Returns:
    int: Total number of comparisons that would be performed
    """
    preprocessed_data = preprocess_data(df)
    total_comparisons = 0

    for i, country_a in enumerate(preprocessed_data):
        values_a = country_a["values"]

        for segment_length in range(min_length, max_length + 1):
            if len(values_a) < segment_length:
                continue

            num_segments_a = len(values_a) - segment_length + 1
            num_segments_b = sum(
                len(country_b["values"]) - segment_length + 1
                for country_b in preprocessed_data[i+1:]
                if len(country_b["values"]) >= segment_length
            )
            total_comparisons += num_segments_a * num_segments_b

    return total_comparisons

class Indicator:
    def __init__(self):
        self.id                   = ""
        self.data                 = pd.DataFrame()
        self.indicator_name       = ""
        self.concept              = ""
        self.license_type         = ""
        self.license_url          = ""
        self.long_definition      = ""
        self.othernotes           = ""
        self.periodicity          = ""
        self.related_source_links = ""
        self.source               = ""
        self.topic                = ""


#### Show Parquet as DF

In [2]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\pattern_maker_output\Afghanistan\patterns.parquet")
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,average_correlation,scdi_count
0,1,198,14,2004,1996,0.805940,2
1,1,198,14,2004,1998,0.750982,2
2,1,198,14,2004,2010,0.706260,2
3,1,217,14,2004,1996,0.897799,5
4,1,217,14,2004,1998,0.884856,5
...,...,...,...,...,...,...,...
15972687,1,155,5,1972,2017,0.865035,1
15972688,1,155,5,1972,2018,0.922607,1
15972689,1,155,5,1972,2019,0.957709,1
15972690,1,158,5,1972,1983,0.999362,1


#### Shrinking

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\indicator_extractors\wb\SP.POP.TOTL\SP.POP.TOTL.parquet")
output_path = r"/workspace/qgi\qgi-4.0\data\indicator_extractors\wb\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet"

# Select 5 countries to keep
target_countries = ["United States", "United Kingdom", "Germany", "France", "Canada", "Australia", "Japan", "India", "China", "Brazil", "Mexico", "South Africa", "Nigeria", "Egypt", "Turkiye", "Russia", "Italy", "Spain", "Argentina", "South Korea", "Indonesia", "Pakistan", "Bangladesh", "Thailand", "Philippines", "Vietnam", "Kenya", "Tanzania", "Uganda", "Ghana", "Senegal", "Morocco", "Algeria", "Israel", "Saudi Arabia", "Iran", "Iraq", "Afghanistan", "Poland", "Hungary", "Czech", "Greece", "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", "Finland", "Switzerland", "Austria"]

# Filter the DataFrame
filtered_df = df[df["country"].isin(target_countries)]

filtered_df.to_parquet(output_path, index = False, compression = "snappy")
print(f"Shrinked Parquet file saved to {output_path}")
filtered_df

Shrinked Parquet file saved to /workspace/qgi\qgi-4.0\data\etl_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked.parquet


,country,year,value
128,Afghanistan,1960,9035043.0
129,Afghanistan,1961,9214083.0
130,Afghanistan,1962,9404406.0
131,Afghanistan,1963,9604487.0
132,Afghanistan,1964,9814318.0
...,...,...,...
16891,South Africa,2019,59587885.0
16892,South Africa,2020,60562381.0
16893,South Africa,2021,61502603.0
16894,South Africa,2022,62378410.0


In [5]:
from typing import Sequence


def process_sequence(seq: Sequence[int]) -> None:
    for item in seq:
        print(item)

# Populate a list and pass it to the function
my_sequence = [1, 2, 3]
process_sequence(my_sequence)

1
2
3


#### Calculating Comaprisons

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\indicator_extractors\wb\NY.GDP.MKTP.KD.ZG\NY.GDP.MKTP.KD.ZG.parquet")

# Example usage:
total = calculate_total_comparisons(df)
print(f"Total comparisons: {total}")

Total comparisons: 60960900


#### Size Saving Comparison

In [ ]:
df = read_parquet_as_df(r"/workspace/qgi\qgi-4.0\data\scdi_finder_output\SP.POP.TOTL_shrinked\SP.POP.TOTL_shrinked_sdcis.parquet")
print(df.head())

   country_a_id  country_b_id  length  start_year_a  start_year_b  correlation
0            45            81       3          1960          1977     0.945475
1            45            81       3          1961          1960     0.970550
2            45            81       3          1961          1961     0.964687
3            45            81       3          1961          1962     0.948679
4            45            81       3          1961          1963     0.965124


In [15]:
df

,country_a_id,country_b_id,length,start_year_a,start_year_b,correlation
0,45,81,3,1960,1977,0.945475
1,45,81,3,1961,1960,0.970550
2,45,81,3,1961,1961,0.964687
3,45,81,3,1961,1962,0.948679
4,45,81,3,1961,1963,0.965124
...,...,...,...,...,...,...
82764,110,236,5,2005,2015,0.958875
82765,110,236,5,2005,2017,0.943587
82766,110,236,5,2005,2018,0.978710
82767,110,236,5,2005,2019,0.901636


In [13]:
df

,country_a,country_b,length,start_year_a,start_year_b,correlation,data_segment_a,data_segment_b
0,China,Germany,3,1960,1977,0.945475,"[667070000.0, 660330000.0, 665770000.0]","[78159814.0, 78091820.0, 78126350.0]"
1,China,Germany,3,1961,1960,0.970550,"[660330000.0, 665770000.0, 682335000.0]","[72814900.0, 73377632.0, 74025784.0]"
2,China,Germany,3,1961,1961,0.964687,"[660330000.0, 665770000.0, 682335000.0]","[73377632.0, 74025784.0, 74714353.0]"
3,China,Germany,3,1961,1962,0.948679,"[660330000.0, 665770000.0, 682335000.0]","[74025784.0, 74714353.0, 75318337.0]"
4,China,Germany,3,1961,1963,0.965124,"[660330000.0, 665770000.0, 682335000.0]","[74714353.0, 75318337.0, 75963695.0]"
...,...,...,...,...,...,...,...,...
82764,Japan,United States,5,2005,2015,0.958875,"[127773000.0, 127854000.0, 128001000.0, 128063...","[320738994.0, 323071755.0, 325122128.0, 326838..."
82765,Japan,United States,5,2005,2017,0.943587,"[127773000.0, 127854000.0, 128001000.0, 128063...","[325122128.0, 326838199.0, 328329953.0, 331526..."
82766,Japan,United States,5,2005,2018,0.978710,"[127773000.0, 127854000.0, 128001000.0, 128063...","[326838199.0, 328329953.0, 331526933.0, 332048..."
82767,Japan,United States,5,2005,2019,0.901636,"[127773000.0, 127854000.0, 128001000.0, 128063...","[328329953.0, 331526933.0, 332048977.0, 333271..."


#### Testing WB Extractor

- PV.EST:             25,016,070 - Political Stability
- NY.GDP.MKTP.CD:     68,162,325 - GDP (current US$)
- SP.DYN.LE00.IN:    217,120,200 - Life expectancy at birth, total (years)
- SE.PRM.ENRR:           936,600 - School enrollment, primary (% gross)
- SL.EMP.TOTL.SP.ZS:  45,988,040 - Employment to population ratio, 15+, total (%) (modeled ILO estimate)
- SL.UEM.TOTL.ZS:     45,988,040 - Unemployment, total (% of total labor force) (modeled ILO estimate)
- NY.GDP.PCAP.CD:     68,162,325 - GDP per capita (current US$)
- MS.MIL.MPRT.KD:      7,033,950 - Arms imports (SIPRI trend indicator values)

In [8]:
class GetIndicatorData:
    def __init__(self, countries_csv_path):
        self.countries_csv_path = countries_csv_path

    def get_indicator_data(self, indicator_id):
        """Fetch and process indicator data from the World Bank"""

        try:
            indicator = self._get_indicator(indicator_id)
            indicator.data = self._remove_all_nan_year_columns(indicator.data)
            indicator.data = self._pivot_dataframe(indicator.data)
            indicator.data = self._rename_columns(indicator.data)
            indicator.data, unmatched_countries = self._clean_country_names(indicator.data)
            indicator.data = self._delete_rows_by_country(indicator.data, unmatched_countries)
            
            print("Possible comparisons: " + str(calculate_total_comparisons(indicator.data)))

            return indicator
        
        except Exception as e:
            print(f"Error fetching or processing indicator data: {str(e)}")
            return None

    def _get_indicator(self, indicator_id):
        """Fetch indicator data from the World Bank API"""
        indicator = Indicator()
        try:
            indicator_metadata = wb.series.metadata.get(indicator_id)
        except Exception as e:
            print(f"Error caught when extracting indicator metadata: {str(e)}")
        indicator.id = indicator_id
        indicator.data = wb.data.DataFrame(indicator_id)
        
        if "IndicatorName" in list(indicator_metadata.metadata.keys()):
            indicator.name = indicator_metadata.metadata["IndicatorName"]
        else:
            indicator.name = ""
        
        indicator.concept = indicator_metadata.concept
        
        if "License_Type" in list(indicator_metadata.metadata.keys()):
            indicator.license_type = indicator_metadata.metadata["License_Type"]
        else:
            indicator.license_type = ""
        
        if "License_URL" in list(indicator_metadata.metadata.keys()):
            indicator.license_url = indicator_metadata.metadata["License_URL"]
        else:
            indicator.license_url = ""
        
        if "Longdefinition" in list(indicator_metadata.metadata.keys()):
            indicator.long_definition = indicator_metadata.metadata["Longdefinition"]
        else:
            indicator.long_definition = ""
        
        if "Othernotes" in list(indicator_metadata.metadata.keys()):
            indicator.othernotes = indicator_metadata.metadata["Othernotes"]
        else:
            indicator.othernotes = ""
        
        if "Periodicity" in list(indicator_metadata.metadata.keys()):
            indicator.periodicity = indicator_metadata.metadata["Periodicity"]
        else:
            indicator.periodicity = ""
        
        if "Relatedsourcelinks" in list(indicator_metadata.metadata.keys()):
            indicator.related_source_links = indicator_metadata.metadata["Relatedsourcelinks"]
        else:
            indicator.related_source_links = ""
        
        if "Source" in list(indicator_metadata.metadata.keys()):
            indicator.source = indicator_metadata.metadata["Source"]
        else:
            indicator.source = ""
        
        if "Topic" in list(indicator_metadata.metadata.keys()):
            indicator.topic = indicator_metadata.metadata["Topic"]
        else:
            indicator.topic = ""
        
        return indicator

    def _remove_all_nan_year_columns(self, df):
        """Remove columns where all values are NaN"""
        for year_col in list(df):
            nan_status = list(set(np.isnan(list(df[year_col]))))
            if len(nan_status) == 1 and str(nan_status[0]) == "True":
                df = df.drop([year_col], axis=1)
        return df

    def _pivot_dataframe(self, df):
        """Pivot the DataFrame from wide to long format"""
        df["economy"] = df.index
        df_long = pd.melt(
            df,
            id_vars=['economy'],
            var_name='year',
            value_name='value'
        )
        df_long['year'] = df_long['year'].str.extract(r'(\d+)').astype(int)
        df_long = df_long.sort_values(by=['economy', 'year']).reset_index(drop=True)
        return df_long

    def _rename_columns(self, df):
        """Rename columns for consistency"""
        column_mapping = {
            'economy': 'country',
        }
        return df.rename(columns=column_mapping)

    def _clean_country_names(self, df):
        """Replace country codes with country names using a reference CSV"""
        # Load reference data
        reference_df = pd.read_csv(self.countries_csv_path)
        reference_df = reference_df.loc[:, ~reference_df.columns.str.contains("Unnamed")]
        
        if 'name_1' not in reference_df.columns:
            raise KeyError("The column 'name_1' is missing in the reference data.")
        
        # Create mapping dictionary
        mapping = {}
        for col in reference_df.columns:
            if col != 'name_1':
                temp_mapping = reference_df.dropna(subset=[col]).set_index(col)['name_1'].to_dict()
                mapping.update(temp_mapping)
        
        # Apply mapping
        df['country'] = df['country'].apply(lambda x: mapping.get(x, x))
        unmatched_values = [
            value for value in df['country'] if value not in mapping.values() and value not in mapping.keys()
        ]
        return df, list(set(unmatched_values))

    def _delete_rows_by_country(self, df, countries_to_delete):
        """Delete rows where the country matches any in the provided list"""
        return df[~df['country'].isin(countries_to_delete)]

ind_getter = GetIndicatorData(COUNTRIES_CSV_PATH)
ind_getter.get_indicator_data("NE.IMP.GNFS.ZS").data

Possible comparisons: 16580025


,country,year,value
0,Aruba,1960,NaN
1,Aruba,1961,NaN
2,Aruba,1962,NaN
3,Aruba,1963,NaN
4,Aruba,1964,NaN
...,...,...,...
17019,Zimbabwe,2019,28.241805
17020,Zimbabwe,2020,25.020302
17021,Zimbabwe,2021,28.071888
17022,Zimbabwe,2022,36.891437
